# City Transportation Data Platform

End-to-end pipeline covering **Phase 1** (ingestion, validation, staging) and **Phase 2** (transformation, integration, business validation).

---

# Phase 1 — Ingest, Validate & Stage

## Task 7 — Build the First Ingestion Program

Load the five raw source files independently with Pandas. Nothing is combined yet — that happens later, at the correct grain, in Phase 2.

In [46]:
import pandas as pd

### Load raw source files

In [47]:
# Load each raw CSV file into its own DataFrame
routes = pd.read_csv('../raw/routes.csv')
vehicles = pd.read_csv('../raw/vehicles.csv')
trips = pd.read_csv('../raw/trips.csv')
passenger_transactions = pd.read_csv('../raw/passenger_transactions.csv')
maintenance = pd.read_csv('../raw/maintenance.csv')

### Quick sanity check
Preview shape and first few rows of each DataFrame.

In [48]:
sources = {
    'routes': routes,
    'vehicles': vehicles,
    'trips': trips,
    'passenger_transactions': passenger_transactions,
    'maintenance': maintenance,
}

for name, df in sources.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

routes: 15 rows x 6 columns
vehicles: 15 rows x 6 columns
trips: 15 rows x 7 columns
passenger_transactions: 15 rows x 6 columns
maintenance: 15 rows x 7 columns


In [49]:
routes.head()

,route_id,route_name,origin,destination,distance_km,expected_duration_min
0,R01,Downtown Express,Central Station,North Hub,18.5,45
1,R02,Crosstown Shuttle,East Mall,West Tech Park,24.0,60
2,R03,Airport Connector,Central Station,Airport Terminal 1,32.0,50
3,R04,University Line,South Metro,State University,12.2,30
4,R05,Harbor Commuter,Port Terminal,Central Station,15.0,35


In [50]:
vehicles.head()

,vehicle_id,license_plate,model,capacity,status,manufacture_year
0,V101,BUS-7890,Volvo B7R,50,Active,2019
1,V102,BUS-7891,Volvo B7R,50,Active,2020
2,V103,BUS-4521,BYD K9,40,Under Maintenance,2021
3,V104,BUS-1209,Scania K320,60,Active,2018
4,V105,BUS-3310,BYD K9,40,Active,2022


In [51]:
trips.head()

,trip_id,route_id,vehicle_id,scheduled_start_time,actual_start_time,actual_end_time,status
0,T1001,R01,V101,2026-03-01 07:00:00,2026-03-01 07:02:00,2026-03-01 07:55:00,Completed
1,T1002,R01,V102,2026-03-01 08:00:00,2026-03-01 08:05:00,2026-03-01 09:05:00,Completed
2,T1003,R02,V104,2026-03-01 07:30:00,2026-03-01 07:30:00,2026-03-01 08:45:00,Completed
3,T1004,R03,V103,2026-03-01 09:00:00,2026-03-01 09:15:00,2026-03-01 10:20:00,Completed
4,T1005,R04,V105,2026-03-01 08:15:00,2026-03-01 08:15:00,2026-03-01 08:43:00,Completed


In [52]:
passenger_transactions.head()

,transaction_id,trip_id,card_id,tap_timestamp,fare_amount,payment_method
0,TX8001,T1001,C-4401,2026-03-01 07:01:15,2.75,SmartCard
1,TX8002,T1001,C-9912,2026-03-01 07:01:40,2.75,Contactless Credit
2,TX8003,T1002,C-1204,2026-03-01 08:03:10,2.75,SmartCard
3,TX8004,T1003,C-8831,2026-03-01 07:28:55,3.50,SmartCard
4,TX8005,T1004,C-5520,2026-03-01 09:12:05,5.00,Mobile Pay


In [53]:
maintenance.head()

,maintenance_id,vehicle_id,service_date,issue_type,description,cost,status
0,M501,V103,2026-02-15,Engine,Overheating issue reported,450.0,Completed
1,M502,V103,2026-02-28,Engine,Coolant leak repair,320.0,Completed
2,M503,V104,2026-02-20,Brakes,Routine brake pad replacement,210.0,Completed
3,M504,V101,2026-02-10,Electrical,Door sensor malfunction,110.0,Completed
4,M505,V103,2026-03-01,Transmission,Gear slippage under load,850.0,In Progress


### Next steps
Each DataFrame (`routes`, `vehicles`, `trips`, `passenger_transactions`, `maintenance`) is now loaded independently and ready for cleaning, validation, and eventual joining in a later step of the pipeline.

---

## Task 9 — Basic Schema Validation

Simple OOP check: does each source match the structure we expect? Nothing is cleaned here — just checked and recorded.

In [54]:
class SchemaValidator:
    """Checks a DataFrame's structure against what we expect. Does not modify the data."""

    def __init__(self, name, df, primary_key, expected_types):
        self.name = name
        self.df = df
        self.primary_key = primary_key
        self.expected_types = expected_types  # {column_name: expected_dtype}

    def check_primary_key(self):
        if self.primary_key not in self.df.columns:
            print(f"[{self.name}] Primary key '{self.primary_key}' is MISSING.")
            return
        nulls = self.df[self.primary_key].isna().sum()
        duplicates = self.df[self.primary_key].duplicated().sum()
        print(f"[{self.name}] Primary key '{self.primary_key}': {nulls} nulls, {duplicates} duplicates.")

    def type_report(self):
        rows = []
        for field, expected_type in self.expected_types.items():
            actual_type = str(self.df[field].dtype) if field in self.df.columns else 'MISSING'
            rows.append({'Field': field, 'Expected Type': expected_type, 'Actual Type': actual_type})
        return pd.DataFrame(rows)

### Expected structure per source

Text columns are expected as `str` (Pandas 3.x's default text dtype).

In [55]:
validators = [
    SchemaValidator('routes', routes, 'route_id', {
        'route_id': 'str', 'route_name': 'str', 'origin': 'str',
        'destination': 'str', 'distance_km': 'float64', 'expected_duration_min': 'int64',
    }),
    SchemaValidator('vehicles', vehicles, 'vehicle_id', {
        'vehicle_id': 'str', 'license_plate': 'str', 'model': 'str',
        'capacity': 'int64', 'status': 'str', 'manufacture_year': 'int64',
    }),
    SchemaValidator('trips', trips, 'trip_id', {
        'trip_id': 'str', 'route_id': 'str', 'vehicle_id': 'str',
        'scheduled_start_time': 'datetime64[ns]', 'actual_start_time': 'datetime64[ns]',
        'actual_end_time': 'datetime64[ns]', 'status': 'str',
    }),
    SchemaValidator('passenger_transactions', passenger_transactions, 'transaction_id', {
        'transaction_id': 'str', 'trip_id': 'str', 'card_id': 'str',
        'tap_timestamp': 'datetime64[ns]', 'fare_amount': 'float64', 'payment_method': 'str',
    }),
    SchemaValidator('maintenance', maintenance, 'maintenance_id', {
        'maintenance_id': 'str', 'vehicle_id': 'str', 'service_date': 'datetime64[ns]',
        'issue_type': 'str', 'description': 'str', 'cost': 'float64', 'status': 'str',
    }),
]

### Primary key check (nulls / duplicates)

In [56]:
for v in validators:
    v.check_primary_key()

[routes] Primary key 'route_id': 0 nulls, 0 duplicates.
[vehicles] Primary key 'vehicle_id': 0 nulls, 0 duplicates.
[trips] Primary key 'trip_id': 0 nulls, 0 duplicates.
[passenger_transactions] Primary key 'transaction_id': 0 nulls, 0 duplicates.
[maintenance] Primary key 'maintenance_id': 0 nulls, 0 duplicates.


### Field / Expected Type / Actual Type

In [57]:
for v in validators:
    print(f"--- {v.name} ---")
    display(v.type_report())

--- routes ---


,Field,Expected Type,Actual Type
0,route_id,str,object
1,route_name,str,object
2,origin,str,object
3,destination,str,object
4,distance_km,float64,float64
5,expected_duration_min,int64,int64


--- vehicles ---


,Field,Expected Type,Actual Type
0,vehicle_id,str,object
1,license_plate,str,object
2,model,str,object
3,capacity,int64,int64
4,status,str,object
5,manufacture_year,int64,int64


--- trips ---


,Field,Expected Type,Actual Type
0,trip_id,str,object
1,route_id,str,object
2,vehicle_id,str,object
3,scheduled_start_time,datetime64[ns],object
4,actual_start_time,datetime64[ns],object
5,actual_end_time,datetime64[ns],object
6,status,str,object


--- passenger_transactions ---


,Field,Expected Type,Actual Type
0,transaction_id,str,object
1,trip_id,str,object
2,card_id,str,object
3,tap_timestamp,datetime64[ns],object
4,fare_amount,float64,float64
5,payment_method,str,object


--- maintenance ---


,Field,Expected Type,Actual Type
0,maintenance_id,str,object
1,vehicle_id,str,object
2,service_date,datetime64[ns],object
3,issue_type,str,object
4,description,str,object
5,cost,float64,float64
6,status,str,object


---

## Task 10 — Create the Staging Layer

Save an untouched copy of each source into `staging/`, named `stg_<source>.csv`. No cleaning happens here — this just closes out Phase 1:

**OPERATIONAL SOURCES -> RAW -> INGESTION -> STAGING**

In [58]:
staging_sources = {
    'stg_routes.csv': routes,
    'stg_vehicles.csv': vehicles,
    'stg_trips.csv': trips,
    'stg_passenger_transactions.csv': passenger_transactions,
    'stg_maintenance.csv': maintenance,
}

for filename, df in staging_sources.items():
    df.to_csv(f'../staging/{filename}', index=False)
    print(f"Saved {filename} ({df.shape[0]} rows x {df.shape[1]} columns)")

Saved stg_routes.csv (15 rows x 6 columns)
Saved stg_vehicles.csv (15 rows x 6 columns)
Saved stg_trips.csv (15 rows x 7 columns)
Saved stg_passenger_transactions.csv (15 rows x 6 columns)
Saved stg_maintenance.csv (15 rows x 7 columns)


---

# Phase 2 — Transform, Validate & Integrate

Picks up where Phase 1 ended: staging data gets reloaded, re-validated, transformed, and integrated into the required analytical outputs.

## Task 1 — Load and Recheck the Staging Layer

Reload the five staging files, build a starting summary (file, rows, columns, primary key), and repeat the primary-key null/duplicate check before any transformation begins.

In [59]:
trans_routes_df = pd.read_csv('../staging/stg_routes.csv')
trans_vehicles_df = pd.read_csv('../staging/stg_vehicles.csv')
trans_trips_df = pd.read_csv('../staging/stg_trips.csv')
trans_passenger_transactions_df = pd.read_csv('../staging/stg_passenger_transactions.csv')
trans_maintenance_df = pd.read_csv('../staging/stg_maintenance.csv')

In [60]:
trans_routes_df.head()

,route_id,route_name,origin,destination,distance_km,expected_duration_min
0,R01,Downtown Express,Central Station,North Hub,18.5,45
1,R02,Crosstown Shuttle,East Mall,West Tech Park,24.0,60
2,R03,Airport Connector,Central Station,Airport Terminal 1,32.0,50
3,R04,University Line,South Metro,State University,12.2,30
4,R05,Harbor Commuter,Port Terminal,Central Station,15.0,35


In [61]:
trans_vehicles_df.head()

,vehicle_id,license_plate,model,capacity,status,manufacture_year
0,V101,BUS-7890,Volvo B7R,50,Active,2019
1,V102,BUS-7891,Volvo B7R,50,Active,2020
2,V103,BUS-4521,BYD K9,40,Under Maintenance,2021
3,V104,BUS-1209,Scania K320,60,Active,2018
4,V105,BUS-3310,BYD K9,40,Active,2022


In [62]:
trans_trips_df.head()

,trip_id,route_id,vehicle_id,scheduled_start_time,actual_start_time,actual_end_time,status
0,T1001,R01,V101,2026-03-01 07:00:00,2026-03-01 07:02:00,2026-03-01 07:55:00,Completed
1,T1002,R01,V102,2026-03-01 08:00:00,2026-03-01 08:05:00,2026-03-01 09:05:00,Completed
2,T1003,R02,V104,2026-03-01 07:30:00,2026-03-01 07:30:00,2026-03-01 08:45:00,Completed
3,T1004,R03,V103,2026-03-01 09:00:00,2026-03-01 09:15:00,2026-03-01 10:20:00,Completed
4,T1005,R04,V105,2026-03-01 08:15:00,2026-03-01 08:15:00,2026-03-01 08:43:00,Completed


In [63]:
trans_passenger_transactions_df.head()

,transaction_id,trip_id,card_id,tap_timestamp,fare_amount,payment_method
0,TX8001,T1001,C-4401,2026-03-01 07:01:15,2.75,SmartCard
1,TX8002,T1001,C-9912,2026-03-01 07:01:40,2.75,Contactless Credit
2,TX8003,T1002,C-1204,2026-03-01 08:03:10,2.75,SmartCard
3,TX8004,T1003,C-8831,2026-03-01 07:28:55,3.50,SmartCard
4,TX8005,T1004,C-5520,2026-03-01 09:12:05,5.00,Mobile Pay


In [64]:
trans_maintenance_df.head()

,maintenance_id,vehicle_id,service_date,issue_type,description,cost,status
0,M501,V103,2026-02-15,Engine,Overheating issue reported,450.0,Completed
1,M502,V103,2026-02-28,Engine,Coolant leak repair,320.0,Completed
2,M503,V104,2026-02-20,Brakes,Routine brake pad replacement,210.0,Completed
3,M504,V101,2026-02-10,Electrical,Door sensor malfunction,110.0,Completed
4,M505,V103,2026-03-01,Transmission,Gear slippage under load,850.0,In Progress


### Starting summary — file, row count, column count, primary key

In [65]:
staging_reload_info = {
    'stg_routes.csv': (trans_routes_df, 'route_id'),
    'stg_vehicles.csv': (trans_vehicles_df, 'vehicle_id'),
    'stg_trips.csv': (trans_trips_df, 'trip_id'),
    'stg_passenger_transactions.csv': (trans_passenger_transactions_df, 'transaction_id'),
    'stg_maintenance.csv': (trans_maintenance_df, 'maintenance_id'),
}

staging_summary = pd.DataFrame([
    {
        'File': filename,
        'Row Count': df.shape[0],
        'Column Count': df.shape[1],
        'Primary Key': pk,
    }
    for filename, (df, pk) in staging_reload_info.items()
])
staging_summary

,File,Row Count,Column Count,Primary Key
0,stg_routes.csv,15,6,route_id
1,stg_vehicles.csv,15,6,vehicle_id
2,stg_trips.csv,15,7,trip_id
3,stg_passenger_transactions.csv,15,6,transaction_id
4,stg_maintenance.csv,15,7,maintenance_id


### Primary-key recheck (nulls / duplicates) before integration

In [66]:
pk_recheck = pd.DataFrame([
    {
        'File': filename,
        'Primary Key': pk,
        'Nulls': int(df[pk].isna().sum()),
        'Duplicates': int(df[pk].duplicated().sum()),
        'Status': 'OK' if not df[pk].isna().sum() and not df[pk].duplicated().sum() else 'ISSUE FOUND',
    }
    for filename, (df, pk) in staging_reload_info.items()
])
pk_recheck

,File,Primary Key,Nulls,Duplicates,Status
0,stg_routes.csv,route_id,0,0,OK
1,stg_vehicles.csv,vehicle_id,0,0,OK
2,stg_trips.csv,trip_id,0,0,OK
3,stg_passenger_transactions.csv,transaction_id,0,0,OK
4,stg_maintenance.csv,maintenance_id,0,0,OK


---

## Task 2 — Apply Required Transformations

Perform only the transformations needed to make the sources joinable and analytically usable: correct data types first, then the required derived fields.

### Required transformations (per the assignment brief)

| Transformation / Derived Field | Expected Work |
| :--- | :--- |
| **Date/time fields** | Convert `trip_date`, `departure_time`, `arrival_time`, `transaction_time`, `service_date`, and `acquisition_date` to appropriate date/time types where possible. |
| **Numerical fields** | Ensure `capacity`, `distance_km`, `expected_duration`, `fare_amount`, `cost`, and `odometer` are numeric. |
| **passenger_count** | Count passenger transactions per trip. |
| **fare_revenue** | Sum `fare_amount` per trip. |
| **actual_duration_minutes** | Compute actual trip duration from `departure_time` and `arrival_time`. |
| **delay_minutes** | `actual_duration_minutes` - `expected_duration`. |
| **utilization_pct** | `passenger_count` / `vehicle capacity` × 100. Explain any value above 100%. |

### Task 2a — Type Conversion

Reusing the same idea as Phase 1's `SchemaValidator`, but this version actually converts each field instead of just reporting on it.

In [67]:
class SchemaTransform:

    def __init__(self, name: str, df: pd.DataFrame, expected_types: dict):
        self.name = name
        self.df = df
        self.expected_types = expected_types

    def transforms_schema(self):
        for field, expected_type in self.expected_types.items():
            if expected_type == "datetime64[ns]":
                self.df[field] = pd.to_datetime(self.df[field])

            elif expected_type == "int64":
                # Preserves nulls using Pandas nullable integer type
                self.df[field] = pd.to_numeric(self.df[field], errors="coerce").astype(
                    "Int64"
                )

            elif expected_type in ["str", "string"]:
                # Preserves nulls as pd.NA instead of converting them to "nan"
                self.df[field] = self.df[field].astype("string")

            else:
                self.df[field] = self.df[field].astype(expected_type)
                
    def type_report(self):
            rows = []
            for field, expected_type in self.expected_types.items():
                actual_type = str(self.df[field].dtype) if field in self.df.columns else 'MISSING'
                rows.append({'Field': field, 'Expected Type': expected_type, 'Actual Type': actual_type})
            return pd.DataFrame(rows)

Each source gets a dictionary mapping its fields to the data type they should be converted to:

In [68]:
transform = [
    SchemaTransform('routes', trans_routes_df, {
        'route_id': 'str', 'route_name': 'str', 'origin': 'str',
        'destination': 'str', 'distance_km': 'float64', 'expected_duration_min': 'int64',
    }),
    SchemaTransform('vehicles', trans_vehicles_df, {
        'vehicle_id': 'str', 'license_plate': 'str', 'model': 'str',
        'capacity': 'int64', 'status': 'str', 'manufacture_year': 'int64',
    }),
    SchemaTransform('trips', trans_trips_df, {
        'trip_id': 'str', 'route_id': 'str', 'vehicle_id': 'str',
        'scheduled_start_time': 'datetime64[ns]', 'actual_start_time': 'datetime64[ns]',
        'actual_end_time': 'datetime64[ns]', 'status': 'str',
    }),
    SchemaTransform('passenger_transactions', trans_passenger_transactions_df, {
        'transaction_id': 'str', 'trip_id': 'str', 'card_id': 'str',
        'tap_timestamp': 'datetime64[ns]', 'fare_amount': 'float64', 'payment_method': 'str',
    }),
    SchemaTransform('maintenance', trans_maintenance_df, {
        'maintenance_id': 'str', 'vehicle_id': 'str', 'service_date': 'datetime64[ns]',
        'issue_type': 'str', 'description': 'str', 'cost': 'float64', 'status': 'str',
    }),
]

Run the transformation:

In [69]:
for t in transform:
    t.transforms_schema()
    print(f"✅ Schema successfully updated in-place: {t.name}")

✅ Schema successfully updated in-place: routes
✅ Schema successfully updated in-place: vehicles
✅ Schema successfully updated in-place: trips
✅ Schema successfully updated in-place: passenger_transactions
✅ Schema successfully updated in-place: maintenance


Verify the conversion by checking each field's resulting dtype:

In [70]:
for t in transform:
    print(f"--- {t.name} ---")
    display(t.type_report())

--- routes ---


,Field,Expected Type,Actual Type
0,route_id,str,string
1,route_name,str,string
2,origin,str,string
3,destination,str,string
4,distance_km,float64,float64
5,expected_duration_min,int64,Int64


--- vehicles ---


,Field,Expected Type,Actual Type
0,vehicle_id,str,string
1,license_plate,str,string
2,model,str,string
3,capacity,int64,Int64
4,status,str,string
5,manufacture_year,int64,Int64


--- trips ---


,Field,Expected Type,Actual Type
0,trip_id,str,string
1,route_id,str,string
2,vehicle_id,str,string
3,scheduled_start_time,datetime64[ns],datetime64[ns]
4,actual_start_time,datetime64[ns],datetime64[ns]
5,actual_end_time,datetime64[ns],datetime64[ns]
6,status,str,string


--- passenger_transactions ---


,Field,Expected Type,Actual Type
0,transaction_id,str,string
1,trip_id,str,string
2,card_id,str,string
3,tap_timestamp,datetime64[ns],datetime64[ns]
4,fare_amount,float64,float64
5,payment_method,str,string


--- maintenance ---


,Field,Expected Type,Actual Type
0,maintenance_id,str,string
1,vehicle_id,str,string
2,service_date,datetime64[ns],datetime64[ns]
3,issue_type,str,string
4,description,str,string
5,cost,float64,float64
6,status,str,string


### Task 2b — Add Derived Fields

No new DataFrame is created here — every derived field is added as a new column directly onto an
existing source's DataFrame. Two of the five fields can be computed from a single source alone;
the other two need one small value pulled in from a parent source first.

| Derived field | Needs | Single source? |
|---|---|---|
| `actual_duration_minutes` | `actual_start_time`, `actual_end_time` | Yes — `trips` alone |
| `passenger_count`, `fare_revenue` | `trip_id`, `transaction_id`, `fare_amount` | Yes — `passenger_transactions` alone (aggregated to trip grain) |
| `delay_minutes` | `actual_duration_minutes` (trips) + `expected_duration_min` (routes) | No — needs `routes` |
| `utilization_pct` | `passenger_count` (from above) + `capacity` (vehicles) | No — needs `vehicles` |

#### Stage 1a — `actual_duration_minutes` (single source: trips)

In [71]:
trans_trips_df['actual_duration_minutes'] = (
    trans_trips_df['actual_end_time'] - trans_trips_df['actual_start_time']
).dt.total_seconds() / 60

# Cancelled trips have no actual start/end time, so duration is correctly left as NaN
trans_trips_df[['trip_id', 'status', 'actual_start_time', 'actual_end_time', 'actual_duration_minutes']].head()

,trip_id,status,actual_start_time,actual_end_time,actual_duration_minutes
0,T1001,Completed,2026-03-01 07:02:00,2026-03-01 07:55:00,53.0
1,T1002,Completed,2026-03-01 08:05:00,2026-03-01 09:05:00,60.0
2,T1003,Completed,2026-03-01 07:30:00,2026-03-01 08:45:00,75.0
3,T1004,Completed,2026-03-01 09:15:00,2026-03-01 10:20:00,65.0
4,T1005,Completed,2026-03-01 08:15:00,2026-03-01 08:43:00,28.0


#### Stage 1b — `passenger_count` / `fare_revenue` (single source: passenger_transactions)

`passenger_transactions` is at *transaction* grain, not *trip* grain, so these two fields can't be added as
plain columns on that table as-is — they only make sense once collapsed down to one row per trip.
`groupby` does that collapse using only fields already inside `passenger_transactions` itself; no other
source is involved yet.

In [72]:
passenger_summary = (
    trans_passenger_transactions_df
    .groupby('trip_id')
    .agg(
        passenger_count=('transaction_id', 'count'),
        fare_revenue=('fare_amount', 'sum'),
    )
    .reset_index()
)
passenger_summary.head()

,trip_id,passenger_count,fare_revenue
0,T1001,2,5.50
1,T1002,1,2.75
2,T1003,1,3.50
3,T1004,1,5.00
4,T1005,1,2.25


#### Stage 2a — bring `passenger_count` / `fare_revenue` onto trips

This is the first cross-source step: joining the trip-level summary above onto `trans_trips_df` directly,
in place. Trips with no matching transactions (e.g. the cancelled trip) get `0`, not a null.

In [73]:
rows_before = len(trans_trips_df)

trans_trips_df = trans_trips_df.merge(passenger_summary, on='trip_id', how='left')
trans_trips_df['passenger_count'] = trans_trips_df['passenger_count'].fillna(0).astype('Int64')
trans_trips_df['fare_revenue'] = trans_trips_df['fare_revenue'].fillna(0.0)

print(f"Rows before join: {rows_before} | Rows after join: {len(trans_trips_df)}")
assert len(trans_trips_df) == rows_before, "Trip grain was NOT preserved by this join!"
trans_trips_df[['trip_id', 'status', 'passenger_count', 'fare_revenue']].head()

Rows before join: 15 | Rows after join: 15


,trip_id,status,passenger_count,fare_revenue
0,T1001,Completed,2,5.50
1,T1002,Completed,1,2.75
2,T1003,Completed,1,3.50
3,T1004,Completed,1,5.00
4,T1005,Completed,1,2.25


#### Stage 2b — `delay_minutes` (needs `routes.expected_duration_min`)

In [74]:
rows_before = len(trans_trips_df)

trans_trips_df = trans_trips_df.merge(
    trans_routes_df[['route_id', 'expected_duration_min']],
    on='route_id',
    how='left',
)
trans_trips_df['delay_minutes'] = trans_trips_df['actual_duration_minutes'] - trans_trips_df['expected_duration_min']

print(f"Rows before join: {rows_before} | Rows after join: {len(trans_trips_df)}")
assert len(trans_trips_df) == rows_before, "Trip grain was NOT preserved by this join!"
trans_trips_df[['trip_id', 'actual_duration_minutes', 'expected_duration_min', 'delay_minutes']].head()

Rows before join: 15 | Rows after join: 15


,trip_id,actual_duration_minutes,expected_duration_min,delay_minutes
0,T1001,53.0,45,8.0
1,T1002,60.0,45,15.0
2,T1003,75.0,60,15.0
3,T1004,65.0,50,15.0
4,T1005,28.0,30,-2.0


#### Stage 2c — `utilization_pct` (needs `vehicles.capacity`)

In [75]:
rows_before = len(trans_trips_df)

trans_trips_df = trans_trips_df.merge(
    trans_vehicles_df[['vehicle_id', 'capacity']],
    on='vehicle_id',
    how='left',
)
trans_trips_df['utilization_pct'] = (trans_trips_df['passenger_count'] / trans_trips_df['capacity']) * 100

print(f"Rows before join: {rows_before} | Rows after join: {len(trans_trips_df)}")
assert len(trans_trips_df) == rows_before, "Trip grain was NOT preserved by this join!"

over_capacity = trans_trips_df[trans_trips_df['utilization_pct'] > 100]
print(f"\nTrips over 100% utilization: {len(over_capacity)}")
trans_trips_df[['trip_id', 'passenger_count', 'capacity', 'utilization_pct']].head()

Rows before join: 15 | Rows after join: 15

Trips over 100% utilization: 0


,trip_id,passenger_count,capacity,utilization_pct
0,T1001,2,50,4.0
1,T1002,1,50,2.0
2,T1003,1,60,1.666667
3,T1004,1,40,2.5
4,T1005,1,40,2.5


**On utilization above 100%:** `passenger_count` counts fare *taps*, not unique seated riders — standees,
multiple taps per boarding, or a trip briefly exceeding its rated seated capacity can all push this over 100%.
It's not necessarily a data error, but any trip over 100% is worth flagging for review rather than assuming
the seat capacity or the tap count is wrong.

#### `trans_trips_df` now holds all five derived fields

In [76]:
trans_trips_df[[
    'trip_id', 'route_id', 'vehicle_id', 'status',
    'passenger_count', 'fare_revenue',
    'actual_duration_minutes', 'expected_duration_min', 'delay_minutes',
    'capacity', 'utilization_pct',
]]

,trip_id,route_id,vehicle_id,status,passenger_count,fare_revenue,actual_duration_minutes,expected_duration_min,delay_minutes,capacity,utilization_pct
0,T1001,R01,V101,Completed,2,5.50,53.0,45,8.0,50,4.0
1,T1002,R01,V102,Completed,1,2.75,60.0,45,15.0,50,2.0
2,T1003,R02,V104,Completed,1,3.50,75.0,60,15.0,60,1.666667
3,T1004,R03,V103,Completed,1,5.00,65.0,50,15.0,40,2.5
4,T1005,R04,V105,Completed,1,2.25,28.0,30,-2.0,40,2.5
5,T1006,R05,V106,Completed,1,2.50,37.0,35,2.0,50,2.0
6,T1007,R06,V107,Completed,1,2.00,22.0,20,2.0,60,1.666667
7,T1008,R07,V108,Cancelled,0,0.00,NaN,55,<NA>,55,0.0
8,T1009,R08,V109,Completed,1,2.25,27.0,25,2.0,40,2.5
9,T1010,R09,V110,Completed,1,3.00,43.0,40,3.0,50,2.0


---

## Task 3 — Referential-Integrity Tests

Check whether each foreign key actually resolves to a real record in its parent source.
**Orphans are reported, never silently dropped.**

| Rule | Child table | FK column | Parent table | PK column |
|---|---|---|---|---|
| 1 | `trips` | `vehicle_id` | `vehicles` | `vehicle_id` |
| 2 | `trips` | `route_id` | `routes` | `route_id` |
| 3 | `passenger_transactions` | `trip_id` | `trips` | `trip_id` |
| 4 | `maintenance` | `vehicle_id` | `vehicles` | `vehicle_id` |

In [77]:
class ForeignKeyValidator:
    """Checks whether a child table's foreign key values all exist in a parent table's
    primary key. Never modifies either DataFrame - only reports what it finds."""

    def __init__(self, rule_name, child_df, fk_column, parent_df, pk_column):
        self.rule_name = rule_name
        self.child_df = child_df
        self.fk_column = fk_column
        self.parent_df = parent_df
        self.pk_column = pk_column

    def find_orphans(self):
        valid_keys = set(self.parent_df[self.pk_column])
        is_orphan = ~self.child_df[self.fk_column].isin(valid_keys)
        return self.child_df[is_orphan]

    def report_row(self):
        orphans = self.find_orphans()
        affected = len(orphans)
        status = (
            'Reported for review; not silently removed'
            if affected else
            'No orphan foreign keys found'
        )
        return {
            'Rule': f'{self.fk_column} -> {self.pk_column}',
            'Source': self.rule_name,
            'Affected Rows': affected,
            'Status / Action': status,
        }

In [78]:
fk_checks = [
    ForeignKeyValidator('trips.vehicle_id -> vehicles.vehicle_id', trans_trips_df, 'vehicle_id', trans_vehicles_df, 'vehicle_id'),
    ForeignKeyValidator('trips.route_id -> routes.route_id', trans_trips_df, 'route_id', trans_routes_df, 'route_id'),
    ForeignKeyValidator('passenger_transactions.trip_id -> trips.trip_id', trans_passenger_transactions_df, 'trip_id', trans_trips_df, 'trip_id'),
    ForeignKeyValidator('maintenance.vehicle_id -> vehicles.vehicle_id', trans_maintenance_df, 'vehicle_id', trans_vehicles_df, 'vehicle_id'),
]

### Orphan records found per rule (listed, not deleted)

In [79]:
for check in fk_checks:
    orphans = check.find_orphans()
    print(f"--- {check.rule_name} ---")
    if orphans.empty:
        print("No orphan records found.\n")
    else:
        display(orphans)

--- trips.vehicle_id -> vehicles.vehicle_id ---
No orphan records found.

--- trips.route_id -> routes.route_id ---
No orphan records found.

--- passenger_transactions.trip_id -> trips.trip_id ---
No orphan records found.

--- maintenance.vehicle_id -> vehicles.vehicle_id ---
No orphan records found.



### Data-quality report

In [80]:
fk_quality_report = pd.DataFrame([check.report_row() for check in fk_checks])
fk_quality_report

,Rule,Source,Affected Rows,Status / Action
0,vehicle_id -> vehicle_id,trips.vehicle_id -> vehicles.vehicle_id,0,No orphan foreign keys found
1,route_id -> route_id,trips.route_id -> routes.route_id,0,No orphan foreign keys found
2,trip_id -> trip_id,passenger_transactions.trip_id -> trips.trip_id,0,No orphan foreign keys found
3,vehicle_id -> vehicle_id,maintenance.vehicle_id -> vehicles.vehicle_id,0,No orphan foreign keys found


---

## Task 4 — Build the Integrated Output at the Correct Grain

**Intended grain: one row per trip.** The passenger-transaction aggregate and the `expected_duration_min` /
`capacity` lookups were already joined onto `trans_trips_df` back in Task 2b. What's left is bringing in the
remaining descriptive attributes from `routes` and `vehicles` to finish `trip_operations_integrated`.

Maintenance is a separate one-to-many relationship against vehicles (not trips), so it is summarized on its
own in Task 7 rather than joined in here.

In [81]:
print(
    f"Intended grain: one row per trip_id "
    f"({trans_trips_df['trip_id'].nunique()} unique trip_id values, {len(trans_trips_df)} rows currently)."
)

Intended grain: one row per trip_id (15 unique trip_id values, 15 rows currently).


### Join in route attributes (route_name, origin, destination, distance_km)

In [82]:
rows_before = len(trans_trips_df)

trip_operations_integrated = trans_trips_df.merge(
    trans_routes_df[['route_id', 'route_name', 'origin', 'destination', 'distance_km']],
    on='route_id',
    how='left',
)

print(f"Rows before join: {rows_before} | Rows after join: {len(trip_operations_integrated)}")
assert len(trip_operations_integrated) == rows_before, "Grain broken by the routes join!"

Rows before join: 15 | Rows after join: 15


### Join in vehicle attributes (license_plate, model, manufacture_year, vehicle status)

In [83]:
rows_before = len(trip_operations_integrated)

# Renamed to vehicle_status to avoid colliding with the trip's own 'status' column
vehicle_attrs = trans_vehicles_df[
    ['vehicle_id', 'license_plate', 'model', 'manufacture_year', 'status']
].rename(columns={'status': 'vehicle_status'})

trip_operations_integrated = trip_operations_integrated.merge(vehicle_attrs, on='vehicle_id', how='left')

print(f"Rows before join: {rows_before} | Rows after join: {len(trip_operations_integrated)}")
assert len(trip_operations_integrated) == rows_before, "Grain broken by the vehicles join!"

Rows before join: 15 | Rows after join: 15


### Final column order + grain confirmation

In [84]:
trip_operations_integrated = trip_operations_integrated[[
    'trip_id', 'route_id', 'route_name', 'origin', 'destination', 'distance_km', 'expected_duration_min',
    'vehicle_id', 'license_plate', 'model', 'manufacture_year', 'capacity', 'vehicle_status',
    'scheduled_start_time', 'actual_start_time', 'actual_end_time', 'status',
    'passenger_count', 'fare_revenue', 'actual_duration_minutes', 'delay_minutes', 'utilization_pct',
]]

assert trip_operations_integrated['trip_id'].is_unique, "trip_id is not unique - grain violated!"
print(f"Grain confirmed: {len(trip_operations_integrated)} rows, {trip_operations_integrated['trip_id'].nunique()} unique trip_id values.")
trip_operations_integrated.head()

Grain confirmed: 15 rows, 15 unique trip_id values.


,trip_id,route_id,route_name,origin,destination,distance_km,expected_duration_min,vehicle_id,license_plate,model,...,vehicle_status,scheduled_start_time,actual_start_time,actual_end_time,status,passenger_count,fare_revenue,actual_duration_minutes,delay_minutes,utilization_pct
0,T1001,R01,Downtown Express,Central Station,North Hub,18.5,45,V101,BUS-7890,Volvo B7R,...,Active,2026-03-01 07:00:00,2026-03-01 07:02:00,2026-03-01 07:55:00,Completed,2,5.50,53.0,8.0,4.0
1,T1002,R01,Downtown Express,Central Station,North Hub,18.5,45,V102,BUS-7891,Volvo B7R,...,Active,2026-03-01 08:00:00,2026-03-01 08:05:00,2026-03-01 09:05:00,Completed,1,2.75,60.0,15.0,2.0
2,T1003,R02,Crosstown Shuttle,East Mall,West Tech Park,24.0,60,V104,BUS-1209,Scania K320,...,Active,2026-03-01 07:30:00,2026-03-01 07:30:00,2026-03-01 08:45:00,Completed,1,3.50,75.0,15.0,1.666667
3,T1004,R03,Airport Connector,Central Station,Airport Terminal 1,32.0,50,V103,BUS-4521,BYD K9,...,Under Maintenance,2026-03-01 09:00:00,2026-03-01 09:15:00,2026-03-01 10:20:00,Completed,1,5.00,65.0,15.0,2.5
4,T1005,R04,University Line,South Metro,State University,12.2,30,V105,BUS-3310,BYD K9,...,Active,2026-03-01 08:15:00,2026-03-01 08:15:00,2026-03-01 08:43:00,Completed,1,2.25,28.0,-2.0,2.5


---

## Task 5 — Validate the Integration

Row counts before/after each join were already printed in Task 4. What's left: confirm uniqueness at the
chosen grain, check whether either join introduced any unexpected nulls, and scan the derived fields for
impossible values. Nothing gets silently corrected here - anything found gets recorded in the report below.

In [85]:
validation_rows = []

# --- uniqueness at grain ---
dup_trip_ids = trip_operations_integrated['trip_id'].duplicated().sum()
validation_rows.append({
    'Check': 'trip_id uniqueness at trip grain',
    'Affected Rows': int(dup_trip_ids),
    'Status / Action': 'Reported for review; not silently corrected' if dup_trip_ids else 'No issues found',
})

# --- missing values introduced by the routes/vehicles joins ---
joined_columns = ['route_name', 'origin', 'destination', 'distance_km',
                   'license_plate', 'model', 'manufacture_year', 'vehicle_status']
for col in joined_columns:
    missing = int(trip_operations_integrated[col].isna().sum())
    validation_rows.append({
        'Check': f'Missing values introduced in {col} after join',
        'Affected Rows': missing,
        'Status / Action': 'Reported for review; not silently corrected' if missing else 'No issues found',
    })

# --- impossible derived-field values ---
impossible_checks = {
    'actual_duration_minutes < 0': trip_operations_integrated['actual_duration_minutes'] < 0,
    'delay_minutes computed but actual_duration_minutes is null': trip_operations_integrated['actual_duration_minutes'].isna() & trip_operations_integrated['delay_minutes'].notna(),
    'fare_revenue < 0': trip_operations_integrated['fare_revenue'] < 0,
    'passenger_count < 0': trip_operations_integrated['passenger_count'] < 0,
    'utilization_pct < 0': trip_operations_integrated['utilization_pct'] < 0,
}
for check_name, mask in impossible_checks.items():
    affected = int(mask.sum())
    validation_rows.append({
        'Check': check_name,
        'Affected Rows': affected,
        'Status / Action': 'Reported for review; not silently corrected' if affected else 'No issues found',
    })

integration_validation_report = pd.DataFrame(validation_rows)
integration_validation_report

,Check,Affected Rows,Status / Action
0,trip_id uniqueness at trip grain,0,No issues found
1,Missing values introduced in route_name after ...,0,No issues found
2,Missing values introduced in origin after join,0,No issues found
3,Missing values introduced in destination after...,0,No issues found
4,Missing values introduced in distance_km after...,0,No issues found
5,Missing values introduced in license_plate aft...,0,No issues found
6,Missing values introduced in model after join,0,No issues found
7,Missing values introduced in manufacture_year ...,0,No issues found
8,Missing values introduced in vehicle_status af...,0,No issues found
9,actual_duration_minutes < 0,0,No issues found


---

## Task 6 — Run Business Validation Queries

Use `trip_operations_integrated` (and `vehicle_maintenance_summary`, built below) to answer these.
Cells left blank on purpose - fill these in yourself.

**1. Top three routes by total `passenger_count` and total `fare_revenue`.**

In [86]:
route_totals = (
    trip_operations_integrated
    .groupby(['route_id', 'route_name'])
    .agg(total_passenger_count=('passenger_count', 'sum'),
         total_fare_revenue=('fare_revenue', 'sum'))
    .reset_index()
)

print("Top 3 by total_passenger_count:")
display(route_totals.sort_values('total_passenger_count', ascending=False).head(3))

print("Top 3 by total_fare_revenue:")
display(route_totals.sort_values('total_fare_revenue', ascending=False).head(3))

Top 3 by total_passenger_count:


,route_id,route_name,total_passenger_count,total_fare_revenue
0,R01,Downtown Express,3,8.25
1,R02,Crosstown Shuttle,1,3.50
2,R03,Airport Connector,1,5.00


Top 3 by total_fare_revenue:


,route_id,route_name,total_passenger_count,total_fare_revenue
0,R01,Downtown Express,3,8.25
2,R03,Airport Connector,1,5.00
12,R13,Suburb Rapid,1,4.50


**2. Trips with positive `delay_minutes`, largest delays first.**

In [87]:
delayed_trips = trip_operations_integrated[trip_operations_integrated['delay_minutes'] > 0]
delayed_trips = delayed_trips.sort_values('delay_minutes', ascending=False)

delayed_trips[['trip_id', 'route_id', 'vehicle_id', 'actual_duration_minutes', 'expected_duration_min', 'delay_minutes']]

,trip_id,route_id,vehicle_id,actual_duration_minutes,expected_duration_min,delay_minutes
1,T1002,R01,V102,60.0,45,15.0
3,T1004,R03,V103,65.0,50,15.0
2,T1003,R02,V104,75.0,60,15.0
0,T1001,R01,V101,53.0,45,8.0
12,T1013,R12,V114,57.0,50,7.0
13,T1014,R13,V115,70.0,65,5.0
9,T1010,R09,V110,43.0,40,3.0
5,T1006,R05,V106,37.0,35,2.0
6,T1007,R06,V107,22.0,20,2.0
11,T1012,R11,V112,17.0,15,2.0


### Build `vehicle_maintenance_summary` (needed for Q3, one row per vehicle)

In [88]:
vehicle_maintenance_summary = (
    trans_maintenance_df
    .groupby('vehicle_id')
    .agg(
        maintenance_count=('maintenance_id', 'count'),
        total_maintenance_cost=('cost', 'sum'),
        latest_service_date=('service_date', 'max'),
        completed_count=('status', lambda s: (s == 'Completed').sum()),
        pending_count=('status', lambda s: (s != 'Completed').sum()),
    )
    .reset_index()
)
vehicle_maintenance_summary

,vehicle_id,maintenance_count,total_maintenance_cost,latest_service_date,completed_count,pending_count
0,V101,1,110.0,2026-02-10,1,0
1,V102,1,600.0,2026-01-15,1,0
2,V103,3,1620.0,2026-03-01,2,1
3,V104,1,210.0,2026-02-20,1,0
4,V105,1,150.0,2026-02-01,1,0
5,V106,1,480.0,2026-02-18,1,0
6,V107,1,95.0,2026-01-22,1,0
7,V108,2,2710.0,2026-02-25,1,1
8,V109,1,210.0,2026-02-05,1,0
9,V110,1,180.0,2026-02-14,1,0


**3. Vehicles with the highest `maintenance_count` or `total_maintenance_cost`.**
(`vehicle_maintenance_summary` is built in Task 7 below - use it here.)

In [89]:
print("Top 3 by maintenance_count:")
display(vehicle_maintenance_summary.sort_values('maintenance_count', ascending=False).head(3))

print("Top 3 by total_maintenance_cost:")
display(vehicle_maintenance_summary.sort_values('total_maintenance_cost', ascending=False).head(3))

Top 3 by maintenance_count:


,vehicle_id,maintenance_count,total_maintenance_cost,latest_service_date,completed_count,pending_count
2,V103,3,1620.0,2026-03-01,2,1
7,V108,2,2710.0,2026-02-25,1,1
10,V113,2,810.0,2026-03-02,0,2


Top 3 by total_maintenance_cost:


,vehicle_id,maintenance_count,total_maintenance_cost,latest_service_date,completed_count,pending_count
7,V108,2,2710.0,2026-02-25,1,1
2,V103,3,1620.0,2026-03-01,2,1
10,V113,2,810.0,2026-03-02,0,2


---

## Task 7 — Save the Phase 2 Outputs

Three files go into `output/`, all with `index=False`: `trip_operations_integrated.csv` (built in Task 4),
`vehicle_maintenance_summary.csv` (built earlier, in Task 6, since Task 6's business query needed it first),
and `data_quality_report.csv` (the referential-integrity results from Task 3 combined with the integration
checks from Task 5).

### Build `data_quality_report` (Task 3 + Task 5 results, combined)

In [90]:
fk_part = fk_quality_report.rename(columns={'Rule': 'Check', 'Source': 'Detail'})[
    ['Check', 'Detail', 'Affected Rows', 'Status / Action']
]
fk_part.insert(0, 'Category', 'Referential Integrity (Task 3)')

integration_part = integration_validation_report.copy()
integration_part.insert(1, 'Detail', '')
integration_part.insert(0, 'Category', 'Integration Validation (Task 5)')
integration_part = integration_part[['Category', 'Check', 'Detail', 'Affected Rows', 'Status / Action']]

data_quality_report = pd.concat([fk_part, integration_part], ignore_index=True)
data_quality_report

,Category,Check,Detail,Affected Rows,Status / Action
0,Referential Integrity (Task 3),vehicle_id -> vehicle_id,trips.vehicle_id -> vehicles.vehicle_id,0,No orphan foreign keys found
1,Referential Integrity (Task 3),route_id -> route_id,trips.route_id -> routes.route_id,0,No orphan foreign keys found
2,Referential Integrity (Task 3),trip_id -> trip_id,passenger_transactions.trip_id -> trips.trip_id,0,No orphan foreign keys found
3,Referential Integrity (Task 3),vehicle_id -> vehicle_id,maintenance.vehicle_id -> vehicles.vehicle_id,0,No orphan foreign keys found
4,Integration Validation (Task 5),trip_id uniqueness at trip grain,,0,No issues found
5,Integration Validation (Task 5),Missing values introduced in route_name after ...,,0,No issues found
6,Integration Validation (Task 5),Missing values introduced in origin after join,,0,No issues found
7,Integration Validation (Task 5),Missing values introduced in destination after...,,0,No issues found
8,Integration Validation (Task 5),Missing values introduced in distance_km after...,,0,No issues found
9,Integration Validation (Task 5),Missing values introduced in license_plate aft...,,0,No issues found


### Save all three outputs

In [91]:
trip_operations_integrated.to_csv('../output/trip_operations_integrated.csv', index=False)
vehicle_maintenance_summary.to_csv('../output/vehicle_maintenance_summary.csv', index=False)
data_quality_report.to_csv('../output/data_quality_report.csv', index=False)

print("Saved to ../output/:")
print(f"  trip_operations_integrated.csv  ({len(trip_operations_integrated)} rows)")
print(f"  vehicle_maintenance_summary.csv ({len(vehicle_maintenance_summary)} rows)")
print(f"  data_quality_report.csv         ({len(data_quality_report)} rows)")

Saved to ../output/:
  trip_operations_integrated.csv  (15 rows)
  vehicle_maintenance_summary.csv (11 rows)
  data_quality_report.csv         (18 rows)


---
# Short Reflection

1. Which join in your scenario had the highest risk of creating duplicate or multiplied rows? Why?

The join between trips and passenger_transactions carried the highest risk. This is because passenger_transactions is at transaction grain, which means multiple rows can share the same trip_id which is why joining it directly onto trips would have produced one trip row per matching transaction instead of one row per trip which multiplies the trip table and is not what we want. That's why it was aggregated by trip_id (passenger_count, fare_revenue) before being joined, rather than joined raw. 

2. What foreign-key problem would be most damaging to the final analytical output?

The most damaging foreign key problem would be an orphaned passenger_transactions.trip_id. This is because a fare transaction that doesn't match any real trip would indicate that fare revenue tied to that orphaned transaction would either be silently dropped from the trip-level totals or counted in overall revenue figures without ever being traced to a specific route or vehicle which is worse.

3. Why is the grain of an integrated table important in data engineering?

Grain is incredibly important since the grain defines what one row means. Every derived metric such as fare_revenue, passenger_count, utilization_pct, is only correct because trip_operations_integrated is guaranteed to be one row per trip. If a join accidentally multiplied that to two rows for one trip, fare_revenue would silently double for that trip with no error thrown. The table would still run, still produce numbers, but as we have already learned in OOP a code that runs doesnt always mean it is correct.

4. Which transformation or validation rule would you automate first if this pipeline ran every day?

I would automate the referential-integrity checks from Task 3, the ForeignKeyValidator. This is because it catches the most damaging class of problem which are orphaned foreign keys silently corrupting revenue and utilization figures which was also stated in the second question. Another reason is also because from a business point of view, it is cheap to run.